In [ ]:
# 🎧 Music Data Pipeline

## 🔌 Testing Database Connection
Successfully connecting to PostgreSQL using psycopg2.

## 📥 Loading Spreadsheet
Reading song data using pandas.

## 🧹 Data Cleaning
Handling missing duration values.

## 🗄️ Loading to Database
Inserting cleaned data into PostgreSQL.

## 🔌 Testing Database Connection

This step verifies that we can successfully connect to the PostgreSQL database.

In [1]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"  # ← change this
)

print("✅ Connected to PostgreSQL!")
conn.close()

✅ Connected to PostgreSQL!


Load your Spreadsheet

In [2]:
import pandas as pd

# ← change this to your actual spreadsheet path
df = pd.read_excel(r"C:\Users\evaas\OneDrive\Desktop\music_pipeline\my_playlist.xlsx")

# Show first 5 rows
print(f"✅ Found {len(df)} songs in spreadsheet")
df.head()

✅ Found 43 songs in spreadsheet


,Song Name,Artist,Album,Year,Duration (sec),Genre,Mood,Language
0,Shape of You,Ed Sheeran,Divide,2017,233,Pop,Romantic / Feel Good,English
1,The Climb,Miley Cyrus,The Climb (Single),2009,229,Pop,Motivational / Emotional,English
2,Who Says,Selena Gomez,When the Sun Goes Down,2011,200,Pop,Self-love / Empowering,English
3,We Don’t Talk Anymore,Charlie Puth,Nine Track Mind,2016,217,Pop,Romantic / Melancholic,English
4,Levitating,Dua Lipa,Future Nostalgia,2020,203,Pop / Disco,Party / Energetic,English


Load Spreadsheet into Database

In [3]:
import psycopg2
import pandas as pd

# Connect to database
conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)
cursor = conn.cursor()

# Read spreadsheet
df = pd.read_excel(r"C:\Users\evaas\OneDrive\Desktop\music_pipeline\my_playlist.xlsx")

# Loop through each row and insert into database
count = 0
for _, row in df.iterrows():
    cursor.execute("""
        INSERT INTO raw_local_tracks 
            (title, artist, album, duration_seconds, file_format)
        VALUES 
            (%s, %s, %s, %s, %s)
    """, (
        row.get("title"),
        row.get("artist"),
        row.get("album"),
        row.get("duration_seconds"),
        "spreadsheet"    # marking source as spreadsheet
    ))
    count += 1

conn.commit()
cursor.close()
conn.close()

print(f"✅ {count} songs loaded from spreadsheet!")

✅ 43 songs loaded from spreadsheet!


In [4]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)

# Read the table back as a dataframe
df = pd.read_sql("SELECT * FROM raw_local_tracks", conn)
conn.close()

print(f"Total rows in database: {len(df)}")
df.head(10)


Total rows in database: 43


C:\Users\evaas\AppData\Local\Temp\ipykernel_30264\2330982859.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM raw_local_tracks", conn)


,id,file_path,title,artist,album,duration_seconds,file_format,loaded_at
0,1,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
1,2,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
2,3,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
3,4,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
4,5,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
5,6,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
6,7,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
7,8,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
8,9,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469
9,10,None,None,None,None,None,spreadsheet,2026-04-12 04:04:04.452469


In [5]:
import psycopg2

# 1. Connect to PostgreSQL
conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)

cursor = conn.cursor()

# 2. Clear all data from table (FAST + reset IDs)
cursor.execute("TRUNCATE TABLE raw_local_tracks RESTART IDENTITY;")

# 3. Commit changes
conn.commit()

# 4. Close connection
cursor.close()
conn.close()

print("✅ All data deleted from raw_local_tracks (table still exists)")

✅ All data deleted from raw_local_tracks (table still exists)


In [6]:
import psycopg2
import pandas as pd

# Connect to database
conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)
cursor = conn.cursor()

# Read spreadsheet
df = pd.read_excel(r"C:\Users\evaas\OneDrive\Desktop\music_pipeline\my_playlist.xlsx")

# Remove old spreadsheet rows
cursor.execute("DELETE FROM raw_local_tracks WHERE file_format = 'spreadsheet'")
conn.commit()

count = 0

for _, row in df.iterrows():
    duration = row.get("Duration (sec)")

    # Clean duration
    if pd.isna(duration) or str(duration).strip() in ["?", "❓", "NA", "N/A", ""]:
        duration = None
    else:
        duration = float(duration)

    cursor.execute("""
        INSERT INTO raw_local_tracks 
            (title, artist, album, duration_seconds, file_format)
        VALUES 
            (%s, %s, %s, %s, %s)
    """, (
        row.get("Song Name"),
        row.get("Artist"),
        row.get("Album"),
        duration,
        "spreadsheet"
    ))
    count += 1

conn.commit()
cursor.close()
conn.close()

print(f"✅ {count} songs loaded from spreadsheet!")

✅ 43 songs loaded from spreadsheet!


In [7]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)

# Read the table back as a dataframe
df = pd.read_sql("SELECT * FROM raw_local_tracks", conn)
conn.close()

print(f"Total rows in database: {len(df)}")
df.head(10)


Total rows in database: 43


C:\Users\evaas\AppData\Local\Temp\ipykernel_30264\2330982859.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM raw_local_tracks", conn)


,id,file_path,title,artist,album,duration_seconds,file_format,loaded_at
0,1,None,Shape of You,Ed Sheeran,Divide,233.0,spreadsheet,2026-04-12 04:04:27.342319
1,2,None,The Climb,Miley Cyrus,The Climb (Single),229.0,spreadsheet,2026-04-12 04:04:27.342319
2,3,None,Who Says,Selena Gomez,When the Sun Goes Down,200.0,spreadsheet,2026-04-12 04:04:27.342319
3,4,None,We Don’t Talk Anymore,Charlie Puth,Nine Track Mind,217.0,spreadsheet,2026-04-12 04:04:27.342319
4,5,None,Levitating,Dua Lipa,Future Nostalgia,203.0,spreadsheet,2026-04-12 04:04:27.342319
5,6,None,Cheap Thrills,Sia,This Is Acting,225.0,spreadsheet,2026-04-12 04:04:27.342319
6,7,None,Closer,The Chainsmokers,Collage,244.0,spreadsheet,2026-04-12 04:04:27.342319
7,8,None,Love Me Like You Do,Ellie Goulding,Fifty Shades of Grey (Soundtrack),252.0,spreadsheet,2026-04-12 04:04:27.342319
8,9,None,Blank Space,Taylor Swift,1989,231.0,spreadsheet,2026-04-12 04:04:27.342319
9,10,None,Unstoppable,Sia,This Is Acting,217.0,spreadsheet,2026-04-12 04:04:27.342319


Load Local Music Folder

In [8]:
import os
from mutagen.mp3 import MP3
from mutagen.wave import WAVE
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)
cursor = conn.cursor()

# ← change this to your music folder
MUSIC_FOLDER = r"C:\Users\evaas\OneDrive\Desktop\Music"

count = 0
skipped = 0

for root, dirs, files in os.walk(MUSIC_FOLDER):
    for file in files:

        full_path = os.path.join(root, file)
        title = None
        artist = None
        album = None
        duration = None
        file_format = None

        try:
            if file.endswith(".mp3"):
                audio = MP3(full_path)
                file_format = "mp3"
                duration = round(audio.info.length, 2)
                tags = audio.tags
                if tags:
                    title = str(tags.get("TIT2", file))
                    artist = str(tags.get("TPE1", "Unknown"))
                    album = str(tags.get("TALB", "Unknown"))

            elif file.endswith(".wav"):
                audio = WAVE(full_path)
                file_format = "wav"
                duration = round(audio.info.length, 2)
                title = file  # wav files rarely have metadata

            else:
                skipped += 1
                continue  # skip non-music files

            cursor.execute("""
                INSERT INTO raw_local_tracks
                    (file_path, title, artist, album, 
                     duration_seconds, file_format)
                VALUES (%s, %s, %s, %s, %s, %s)
            """, (
                full_path,
                title or file,
                artist or "Unknown",
                album or "Unknown",
                duration,
                file_format
            ))
            count += 1
            print(f"📂 Loaded: {file}")

        except Exception as e:
            print(f"⚠️ Skipped {file}: {e}")
            skipped += 1

conn.commit()
cursor.close()
conn.close()

print(f"\n✅ Done! {count} songs loaded, {skipped} files skipped")

📂 Loaded: 01 The Climb (Miley Cyrus Cover).mp3
📂 Loaded: 240279-13373381-3c94-4f74-9b21-a50737487e4a.mp3
📂 Loaded: 246929-6baa6069-6b65-440f-87c4-807ad4f97c4c.mp3
⚠️ Skipped 256612-02976106-33b8-457c-9306-047fbba505d1.mp3: can't sync to MPEG frame
📂 Loaded: angga-renggana-t_the-chainsmokers-ft-halsey-closer.mp3
📂 Loaded: Ashe - Moral Of The Story (Lyrics).mp3
📂 Loaded: Believer - Imagine Dragons.mp3
📂 Loaded: Diamond Heart - Alan Walker, Sophia Somajo.mp3
📂 Loaded: Huntrix - Golden (Lyrics) KPop Demon Hunters.mp3
📂 Loaded: Maroon 5-Girls Like You.mp3
📂 Loaded: Princess Celestia - A Thousand Years - Christina Perri (AI Cover) [TubeRipper (mp3cut.net).mp3
📂 Loaded: Rarity Shining Armor - On The Floor - Jennifer Lopez ft (mp3cut.net).mp3
📂 Loaded: rio160703_selena-gomez-and-charlie-puth-we-dont-talk-anymore.mp3
📂 Loaded: selena-gomez_who-says.mp3
📂 Loaded: Señorita - Shawn Mendes.mp3
📂 Loaded: Shakira -  - Waka Waka (Time For Africa).mp3
📂 Loaded: Shakira - Hips Don't Lie (Lyrics) ft. Wyc

Verify Everything in Database

In [9]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    database="music_db",
    user="postgres",
    password="postgres123"
)

# Read the table back as a dataframe
df = pd.read_sql("SELECT * FROM raw_local_tracks", conn)
conn.close()

print(f"Total rows in database: {len(df)}")
df.head(10)

Total rows in database: 62


C:\Users\evaas\AppData\Local\Temp\ipykernel_30264\2926525916.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM raw_local_tracks", conn)


,id,file_path,title,artist,album,duration_seconds,file_format,loaded_at
0,1,NaN,Shape of You,Ed Sheeran,Divide,233.0,spreadsheet,2026-04-12 04:04:27.342319
1,2,NaN,The Climb,Miley Cyrus,The Climb (Single),229.0,spreadsheet,2026-04-12 04:04:27.342319
2,3,NaN,Who Says,Selena Gomez,When the Sun Goes Down,200.0,spreadsheet,2026-04-12 04:04:27.342319
3,4,NaN,We Don’t Talk Anymore,Charlie Puth,Nine Track Mind,217.0,spreadsheet,2026-04-12 04:04:27.342319
4,5,NaN,Levitating,Dua Lipa,Future Nostalgia,203.0,spreadsheet,2026-04-12 04:04:27.342319
5,6,NaN,Cheap Thrills,Sia,This Is Acting,225.0,spreadsheet,2026-04-12 04:04:27.342319
6,7,NaN,Closer,The Chainsmokers,Collage,244.0,spreadsheet,2026-04-12 04:04:27.342319
7,8,NaN,Love Me Like You Do,Ellie Goulding,Fifty Shades of Grey (Soundtrack),252.0,spreadsheet,2026-04-12 04:04:27.342319
8,9,NaN,Blank Space,Taylor Swift,1989,231.0,spreadsheet,2026-04-12 04:04:27.342319
9,10,NaN,Unstoppable,Sia,This Is Acting,217.0,spreadsheet,2026-04-12 04:04:27.342319


In [10]:
df.head(62)

,id,file_path,title,artist,album,duration_seconds,file_format,loaded_at
0,1,NaN,Shape of You,Ed Sheeran,Divide,233.00,spreadsheet,2026-04-12 04:04:27.342319
1,2,NaN,The Climb,Miley Cyrus,The Climb (Single),229.00,spreadsheet,2026-04-12 04:04:27.342319
2,3,NaN,Who Says,Selena Gomez,When the Sun Goes Down,200.00,spreadsheet,2026-04-12 04:04:27.342319
3,4,NaN,We Don’t Talk Anymore,Charlie Puth,Nine Track Mind,217.00,spreadsheet,2026-04-12 04:04:27.342319
4,5,NaN,Levitating,Dua Lipa,Future Nostalgia,203.00,spreadsheet,2026-04-12 04:04:27.342319
...,...,...,...,...,...,...,...,...
57,58,C:\Users\evaas\OneDrive\Desktop\Music\Shakira ...,Waka Waka (Time For Africa),Shakira,Unknown,201.72,mp3,2026-04-12 04:04:43.063290
58,59,C:\Users\evaas\OneDrive\Desktop\Music\Shakira ...,Shakira - Hips Don't Lie (Lyrics) ft. Wyclef J...,Unknown,Unknown,218.23,mp3,2026-04-12 04:04:43.063290
59,60,C:\Users\evaas\OneDrive\Desktop\Music\Soda Pop...,Soda Pop,Saja Boys - Topic,Soda Pop,150.83,mp3,2026-04-12 04:04:43.063290
60,61,C:\Users\evaas\OneDrive\Desktop\Music\SpotiDow...,Taylor Swift - Blank Space (Audio),Taylor Swift Audios,Taylor Swift - Blank Space (Audio),231.16,mp3,2026-04-12 04:04:43.063290
